In [4]:
# 1. Mengambil informasi data koordinat ke-34 provinsi sesuai dengan dataset "komoditas_beras_2022_2026"

import pandas as pd
from geopy.geocoders import Nominatim
import time

provinsi = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Jambi", 
    "Sumatera Selatan", "Bengkulu", "Lampung", "Kepulauan Bangka Belitung", 
    "Kepulauan Riau", "DKI Jakarta", "Jawa Barat", "Jawa Tengah", 
    "DI Yogyakarta", "Jawa Timur", "Banten", "Bali", "Nusa Tenggara Barat", 
    "Nusa Tenggara Timur", "Kalimantan Barat", "Kalimantan Tengah", 
    "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara", 
    "Sulawesi Utara", "Sulawesi Tengah", "Sulawesi Selatan", 
    "Sulawesi Tenggara", "Gorontalo", "Sulawesi Barat", "Maluku", 
    "Maluku Utara", "Papua Barat", "Papua"
]

geolocator = Nominatim(user_agent="laptop_yang_selalu_terbuka")
data_koordinat = []

for prov in provinsi:
    location = geolocator.geocode(prov + ", Indonesia")
    if location:
        data_koordinat.append({
            "Province_Name": prov,
            "Latitude": location.latitude,
            "Longitude": location.longitude
        })
    time.sleep(1)

data_frame_koordinat = pd.DataFrame(data_koordinat)
data_frame_koordinat.to_csv("../data/raw/koordinat_provinsi.csv", index=False)
print("data berhasil disimpan")
print(data_frame_koordinat.head())

data berhasil disimpan
    Province_Name  Latitude   Longitude
0            Aceh  4.368549   97.025302
1  Sumatera Utara  2.192352   99.381220
2  Sumatera Barat -0.582753  100.613338
3            Riau  1.049808  101.654843
4           Jambi -1.706997  102.714004


In [5]:
# 2. Mengambil data cuaca tiap harinya dari NASA POWER API, sejak 2022-01-01 s/d 2026-02-12

import requests
import os
from tqdm import tqdm

def get_nasa_weather(lat, lon, start_date, end_date):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "start": start_date,
        "end": end_date,
        "latitude": lat,
        "longitude": lon,
        "community": "ag",
        "parameters": "T2M,PRECTOTCORR", # T2M=Suhu, PRECTOTCORR=Curah Hujan
        "format": "json",
        "header": "true"
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()['properties']['parameter']
    else:
        return None

# Konfigurasi tanggal sesuai dataset "komoditas_beras_2022_2026.csv"
start = "20220101"
end = "20260212"

data_cuaca = []

for index, row in tqdm(data_frame_koordinat.iterrows(), total=data_frame_koordinat.shape[0]):
    data = get_nasa_weather(row['Latitude'], row['Longitude'], start, end)
    if data:
        # Re-format data ke bentuk tabular
        dates = list(data['T2M'].keys())
        for d in dates:
            data_cuaca.append({
                "Date_Param": d,
                "Province_Name": row['Province_Name'],
                "Temperature": data['T2M'][d],
                "Precipitation": data['PRECTOTCORR'][d]
            })

data_frame_cuaca = pd.DataFrame(data_cuaca)

# Konversi format tanggal NASA (YYYYMMDD) ke format standar
data_frame_cuaca['Date_Param'] = pd.to_datetime(data_frame_cuaca['Date_Param'], format='%Y%m%d')
data_frame_cuaca.to_csv("../data/raw/data_cuaca_2022_2026.csv", index=False)
print("\berhasil")

100%|██████████| 34/34 [00:44<00:00,  1.31s/it]

erhasil
